# LLM Training Notebook

**Flow of this notebook:**

1. **Hugging Face Basics** — load a dataset, print a sample, load a model via a pipeline, run it.
2. **Prompt Engineering** — (almost) the full prompt engineering tutorial, plus the same prompts run via **OpenRouter** on **Gemini** and **Anthropic Claude**.
3. **Basic LangChain Chains** — a minimal prompt → chain → invoke example, using OpenRouter as the LLM backend.
4. **Token Budgeting (tiktoken)** — counting input/output tokens and estimating cost, before we get to fine-tuning.
5. **Fine-Tuning (bare minimum)** — BERT classification fine-tuning, and generation-model fine-tuning with **LoRA** and **QLoRA**.

> Cells are adapted from the original Colab notebooks (Hugging Face tutorial, Prompt Engineering tutorial, and the *Hands-On Large Language Models* chapters 7/11/12), condensed and combined into a single training notebook. Wherever the original code used a direct OpenAI key/client, it's been swapped for an **OpenRouter** client (OpenAI-compatible API) so the same code can call OpenAI, Gemini, or Anthropic models with one key.


## 0. Setup

Installs everything needed for the whole notebook, plus the shared OpenRouter client used in the Prompt Engineering and LangChain sections.

**Paste your OpenRouter API key below** (get one at https://openrouter.ai/keys). The same key works across OpenAI, Google, and Anthropic models on OpenRouter.


In [ ]:
!pip install -q torch transformers>=4.41.2 datasets>=2.18.0 accelerate>=0.31.0 sentencepiece evaluate
!pip install -q langchain>=0.1.17 langchain-openai>=0.1.6 langchain-community
!pip install -q openai>=1.13.3 tiktoken
!pip install -q peft>=0.11.1 bitsandbytes>=0.43.1 trl>=0.9.4


In [ ]:
import os
import torch
from openai import OpenAI

# Auto-detect GPU if available (falls back to CPU where not)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# --- OpenRouter client (OpenAI-compatible API) ---
# Paste your OpenRouter key here, or set it as an environment variable beforehand.
OPENROUTER_API_KEY = "PASTE_YOUR_OPENROUTER_API_KEY_HERE"
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

def chat(model, messages, **kwargs):
    """Call any OpenRouter-hosted model (OpenAI / Gemini / Anthropic / etc.) with the same interface."""
    response = client.chat.completions.create(model=model, messages=messages, **kwargs)
    return response.choices[0].message.content


## Part 1: Hugging Face Basics

Load a dataset, print a sample, load a pretrained model through a `pipeline`, and run it.

*(Adapted from `Tutorial_1_Hugging_Face_Final_Version.ipynb`)*


In [ ]:
from datasets import load_dataset

# Load a dataset and print a sample
dataset = load_dataset("imdb", split="train")
print(dataset[0])


In [ ]:
from transformers import pipeline

# Load a pretrained model via a pipeline and run it
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
)
result = sentiment_pipeline(dataset[0]["text"][:512])
print(result)


## Part 2: Prompt Engineering

*(Adapted from `Tutorial_2_Prompt_Engineering_ver2_1_.ipynb`, kept almost entirely as-is. New: the same zero-shot prompt is also run via OpenRouter on Gemini and Anthropic Claude, right after the original Hugging Face version.)*

### Loading Hugging Face Model


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import textwrap

# Load model and tokenizer
model_name = "microsoft/Phi-3-mini-4k-instruct"
# model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
# model_name = "mistralai/Mistral-7B-Instruct-v0.2"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",  # auto-detects GPU/CPU (was hardcoded "cuda" in the original)
    torch_dtype="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Create a pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=1500,
    do_sample=False,
)


# 1. Zero-shot Prompting: Basic Dish Recommendation

## 1.1 Declaring the Prompt and Running through the pipeline to generate output

In [ ]:
# Prompt
messages = [
    {"role": "user", "content": "Recommend a dish to someone who is hungry and likes spicy food."}
]

# Generate the output
output = pipe(messages)
#print(output[0]["generated_text"])

print(textwrap.fill(output[0]["generated_text"], width=50))


### 1.1.1 Same prompt via OpenRouter — Gemini

New addition: run the exact same prompt through OpenRouter, this time on a Gemini model.


In [ ]:
# Same zero-shot prompt, run via OpenRouter on Gemini
gemini_output = chat(
    model="google/gemini-flash-latest",
    messages=messages,
)
print(gemini_output)


### 1.1.2 Same prompt via OpenRouter — Anthropic Claude

New addition: run the exact same prompt through OpenRouter, this time on an Anthropic Claude model.


In [ ]:
# Same zero-shot prompt, run via OpenRouter on Anthropic Claude
claude_output = chat(
    model="anthropic/claude-sonnet-latest",
    messages=messages,
)
print(claude_output)


## 1.2 Using the apply_chat_template method to convert structured chat messages into a plain text prompt

In [ ]:
# Apply prompt template
messages = [
    {"role": "user", "content": "Recommend a dish to someone who is hungry and likes spicy food."}
]

prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False)
print(prompt)


##1.3 Arguments like temperature and top_p while running the pipeline
### 1.3.1 temperature controls how creative or random the model is when choosing words.


In [ ]:
output = pipe(messages, do_sample=True, temperature=1.9)
print(textwrap.fill(output[0]["generated_text"], width=100))


### 1.3.2 top-p chooses from the smallest group of words that together have p% of the probability.

In [ ]:
output = pipe(messages, do_sample=True, top_p=0.5)
print(textwrap.fill(output[0]["generated_text"], width=100))


### 1.3.3 High Temperature and High p gives creative and unexpected outputs among the large pool of potential tokens

In [ ]:
# Apply prompt template
messages = [
    {"role": "user", "content": "What are some unconventional ways to increase weekday lunch traffic at my restaurant?"}
]

prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False)

output = pipe(messages, do_sample=True, temperature=1.9, top_p=1)
print(textwrap.fill(output[0]["generated_text"], width=100))


### 1.3.4 Low Temperature and Low p gives deterministic output that are predictable and focused. No creativity and no surprises.

In [ ]:
# Apply prompt template
messages = [
    {"role": "user", "content": "Write a message to staff explaining how to handle customer complaints politely and professionally"}
]

prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False)

output = pipe(messages, do_sample=True, temperature=0.1, top_p=0.1)
print(textwrap.fill(output[0]["generated_text"], width=100))


### 1.3.5 Low Temperature and High p gives deterministic output that have a bit more variety. Outputs are deterministic but with highly probable predicted tokens

In [ ]:
# Apply prompt template
messages = [
    {"role": "user", "content": "Write short, appealing descriptions for each dish on our dinner menu"}
]

prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False)

output = pipe(messages, do_sample=True, temperature=0.1, top_p=1)
print(textwrap.fill(output[0]["generated_text"], width=100))


### 1.3.6 High Temperature and Low p gives creative outputs but within a predictable pool of tokens

In [ ]:
# Apply prompt template
messages = [
    {"role": "user", "content": "Give creative but clear names for our classic dishes like grilled chicken, Caesar salad, and chocolate cake"}
]

prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False)


output = pipe(messages, do_sample=True, temperature=1.9, top_p=0.1)
print(textwrap.fill(output[0]["generated_text"], width=100))


##1.4 Assigning Roles in Prompt and declaring Dynamic Prompts

In [ ]:
# Indian Restaurant Review Summarization Prompt

# Sample text (replace this with any real or synthetic review text)
text = """Masala Mahal is a vibrant Indian restaurant located in the heart of the city. The décor blends traditional Indian elements with a modern touch — brass lamps, colorful murals, and soft sitar music set the mood.
The food is authentic and flavorful. Highlights included the butter chicken, which was creamy and well-spiced, and the paneer tikka, grilled to perfection. The garlic naan was fluffy and complemented the curries beautifully.
There were plenty of vegetarian and vegan options, with a separate Jain menu available. Service was courteous and prompt, though during peak hours there was a slight delay in getting appetizers.
The desserts, especially the gulab jamun, were warm and satisfying. Overall, Masala Mahal offers a delightful experience with traditional Indian hospitality."""

# Prompt components
persona = (
    "You are ChefBot, a virtual restaurant assistant created for Indian restaurants. "
    "You specialize in analyzing customer feedback and summarizing it into actionable insights for chefs and staff.\n"
)

instruction = "Summarize the customer review provided.\n"

context = (
    "Your summary should highlight important aspects of the dining experience, especially focusing on food quality, service, ambiance, variety, and any concerns or complaints.\n"
)

data_format = (
    "Format your summary as bullet points under the following categories: "
    "Food, Ambiance, Service, Variety, Drawbacks. "
    "End with a brief paragraph summarizing the customer's overall impression.\n"
)

audience = (
    "The summary is for the restaurant owner and kitchen staff who need to quickly understand what the customer appreciated or disliked.\n"
)

tone = "The tone should be professional, clear, and actionable.\n"

data = f"Customer review to summarize:\n{text}"

# Final Prompt
query = persona + instruction + context + data_format + audience + tone + data


In [ ]:
messages = [
    {"role": "user", "content": query}
]


print(textwrap.fill(tokenizer.apply_chat_template(messages, tokenize=False), width=100))


In [ ]:
# Generate the output
outputs = pipe(messages)
print(textwrap.fill(outputs[0]["generated_text"], width=100))


## 1.5 In-Context Learning: Trying to recognise if the dish is Vegan or not

In [ ]:
one_shot_prompt = [
    {
        "role": "user",
        "content": (
            "Determine if the following dish is vegan-friendly or not.\n\n"
            "Dish: Paneer Butter Masala\n"
            "Description: Indian cottage cheese cubes cooked in a creamy tomato-based gravy with butter and cream.\n"
            "Is it vegan?"
        )
    },
    {
        "role": "assistant",
        "content": "❌ No, it contains dairy (paneer, butter, and cream)."
    },
    {
        "role": "user",
        "content": (
            "Dish: Chana Masala\n"
            "Description: Chickpeas simmered in a spicy onion-tomato gravy with garlic, ginger, and traditional spices.\n"
            "Is it vegan?"
        )
    },
    {
        "role": "assistant",
        "content": "✅ Yes, it is typically vegan as it contains no animal products."
    },
    {
        "role": "user",
        "content": (
            "Dish: Vegetable Biryani\n"
            "Description: Aromatic rice cooked with mixed vegetables, saffron, and spices. Often served with raita.\n"
            "Is it vegan?"
        )
    }
]

# Simulate what the prompt looks like
print(tokenizer.apply_chat_template(one_shot_prompt, tokenize=False))


In [ ]:
# Generate the output
outputs = pipe(one_shot_prompt)
#print(outputs[0]["generated_text"])
print(textwrap.fill(outputs[0]["generated_text"], width=100))


## 1.6 Chain Prompting: Coming up with new creative dish names

In [ ]:
# Generate a name and slogan for a new Indian fusion dish

dish_prompt = [
    {
        "role": "user",
        "content": "Create a name and slogan for a spicy Indian street food dish made with crispy tofu, tangy coriander chutney, and chili flakes."
    }
]

# Assuming you're using a text generation pipeline like HuggingFace's `pipeline("text-generation")`
outputs = pipe(dish_prompt)

dish_description = outputs[0]["generated_text"]
#print(dish_description)

print(textwrap.fill(outputs[0]["generated_text"], width=100))


In [ ]:
# Step 2: Use the generated name and slogan in a follow-up prompt
pitch_prompt = [
    {
        "role": "user",
        "content": f"""Using the following dish name and slogan, write a short, engaging pitch that a waiter or voice assistant could say to a customer:

{dish_description}

Make it sound friendly, mouth-watering, and tempting, in under 60 words."""
    }
]

pitch_output = pipe(pitch_prompt)
pitch_text = pitch_output[0]["generated_text"]

print(textwrap.fill(pitch_text, width=100))

#print("Waiter/Voice Agent Pitch:", pitch_text)


# 2 Reasoning with Generative Models

## 2.1 Chain-of-Thought: Think Before Answering

In [ ]:
# Answering kitchen inventory questions without showing reasoning

kitchen_prompt = [
    {
        "role": "user",
        "content": "We have 10 bags of basmati rice. We ordered 5 more today. How many bags do we have now?"
    },
    {
        "role": "assistant",
        "content": "15"
    },
    {
        "role": "user",
        "content": "There are 30 mangoes in the kitchen. The chef used 12 for smoothies and we received 10 more from the supplier. How many mangoes are there now?"
    }
]

# Run the model (using a text-generation pipeline)
outputs = pipe(kitchen_prompt)

# Print only the generated output
print(outputs[0]["generated_text"])


In [ ]:
# Answering with Chain-of-Thought (Step-by-step reasoning)

chefbot_cot_prompt = [
    {
        "role": "user",
        "content": "We have 10 kg of rice. We used 4 kg to prepare biryani. Then we received 7 kg more in the new delivery. How much rice do we have now?"
    },
    {
        "role": "assistant",
        "content": "We started with 10 kg of rice. 4 kg was used for biryani, so we have 6 kg left. Then we received 7 kg more. 6 + 7 = 13 kg. The answer is 13 kg."
    },
    {
        "role": "user",
        "content": "There were 40 samosas in the kitchen. 25 were sold at lunch, and 15 more were prepared in the evening. How many samosas are there now?"
    }
]

# Run the model
outputs = pipe(chefbot_cot_prompt)
print(outputs[0]["generated_text"])


## 2.3 Zero-shot Chain-of-Thought

In [ ]:
#Zero-Shot Chain-of-Thought (no examples, just "Let's think step-by-step")

chefbot_zeroshot_cot_prompt = [
    {
        "role": "user",
        "content": (
            "We had 15 liters of milk in the fridge. The chef used 9 liters for tea and desserts. "
            "Then we received 5 more liters in the morning delivery. How much milk do we have now? Let's think step-by-step."
        )
    }
]

# Run the model
outputs = pipe(chefbot_zeroshot_cot_prompt)
print(outputs[0]["generated_text"])


# 3 Tree-of-Thought: Exploring Intermediate Steps

In [ ]:
zeroshot_tot_prompt = [
    {
        "role": "user",
        "content": (
            "Imagine three different kitchen assistants are answering this question. "
            "All assistants will write down each step of their thinking, then share it with the group. "
            "Then all assistants will go on to the next step, etc. If any assistant realizes they're wrong at any point, they leave. "
            "The question is: 'We had 18kg of onions. 10kg were used for curry prep, and 4kg arrived in the morning delivery. How many kilograms of onions are there now?' "
            "Make sure to print each step as assistants discuss and agree on the final result."
        )
    }
]


In [ ]:
# Generate the output
outputs = pipe(zeroshot_tot_prompt)
print(outputs[0]["generated_text"])


# 4 Output Verification

## 4.1 Zero-Shot prompt to create a dish profile in JSON

In [ ]:
chefbot_zeroshot_prompt = [
    {
        "role": "user",
        "content": "Create a character profile for a new Indian seven course meal in JSON format."
    }
]

# Run the model
outputs = pipe(chefbot_zeroshot_prompt)
print(outputs[0]["generated_text"])


## 4.2 One Shot Prompt providing Output Structure Example

In [ ]:
one_shot_template = """Create a short dish profile for an Indian restaurant menu. Use only this format:

{
  "name": "DISH NAME",
  "ingredients": ["INGREDIENT 1", "INGREDIENT 2", "..."],
  "spice_level": "MILD, MEDIUM, or HOT",
  "description": "A SHORT DESCRIPTION OF THE DISH"
}
"""

one_shot_prompt = [
    {"role": "user", "content": one_shot_template}
]

# Run the model
outputs = pipe(one_shot_prompt)
print(outputs[0]["generated_text"])


# 5. Grammar: Constrained Sampling
### Grammar Constrained Sampling typically refers to a technique used in natural language generation (NLG) or machine learning models—especially language models—where the generation of text is restricted (or guided) by specific grammatical rules or structures.


In [ ]:
import gc
import torch
del model, tokenizer, pipe

# Flush memory
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# llama-cpp-python is already installed via the Setup cell / requirements.txt
from llama_cpp.llama import Llama

# Load Phi-3
llm = Llama.from_pretrained(
    repo_id="microsoft/Phi-3-mini-4k-instruct-gguf",
    filename="*fp16.gguf",
    n_gpu_layers=-1,
    n_ctx=2048,
    verbose=False
)
# Note: on some setups you may need to restart the kernel after installing
# llama-cpp-python for the changes to take effect.


In [ ]:
# Generate output
output = llm.create_chat_completion(
    messages=[
        {"role": "user", "content": "Create a menu for a Corporate Party."},
    ],
    response_format={"type": "json_object"},
    temperature=0,
)['choices'][0]['message']["content"]


In [ ]:
import json

# Format as json
json_output = json.dumps(json.loads(output), indent=4)
print(json_output)


## Part 3: Basic LangChain Chains

*(Adapted from `Chapter_7_-_Advanced_Text_Generation_Techniques_and_Tools.ipynb`. The original used a local `LlamaCpp` model / a raw `ChatOpenAI(... os.environ["OPENAI_API_KEY"] ...)` call for the agent section; here the chain's LLM is `ChatOpenAI` pointed at OpenRouter, so it can call any OpenRouter model with the same OpenAI-style client.)*


In [ ]:
from langchain_openai import ChatOpenAI

# ChatOpenAI pointed at OpenRouter instead of directly at OpenAI
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",  # any OpenRouter model string works here
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0,
)


In [ ]:
from langchain import PromptTemplate

# Create a prompt template with the "input_prompt" variable
template = "{input_prompt}"
prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt"]
)

# Basic chain: prompt -> llm
basic_chain = prompt | llm


In [ ]:
# Use the chain
basic_chain.invoke(
    {
        "input_prompt": "Hi! My name is Maarten. What is 1 + 1?",
    }
)


## Token Budgeting (tiktoken)

New addition: before fine-tuning, it's worth showing learners how to estimate input/output token counts and cost for LLM calls. `tiktoken` is OpenAI's tokenizer, so counts for Gemini/Claude are an *approximation* (each provider tokenizes slightly differently) — good enough for budgeting, not exact billing.


In [ ]:
import tiktoken

# cl100k_base is the encoding used by GPT-3.5/GPT-4-era OpenAI models;
# it's used here as a reasonable general-purpose approximation for any model.
encoding = tiktoken.get_encoding("cl100k_base")

def estimate_cost(prompt_text, completion_text, model_name="openai/gpt-4o-mini",
                   input_price_per_1k=0.00015, output_price_per_1k=0.0006):
    """
    Rough token count + cost estimate for a single prompt/completion pair.
    Price defaults are illustrative — check OpenRouter's pricing page for the model you use.
    """
    input_tokens = len(encoding.encode(prompt_text))
    output_tokens = len(encoding.encode(completion_text))

    input_cost = (input_tokens / 1000) * input_price_per_1k
    output_cost = (output_tokens / 1000) * output_price_per_1k

    print(f"Model: {model_name}")
    print(f"Input tokens : {input_tokens}  (~${input_cost:.6f})")
    print(f"Output tokens: {output_tokens}  (~${output_cost:.6f})")
    print(f"Total cost   : ~${input_cost + output_cost:.6f}")

    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "estimated_cost_usd": input_cost + output_cost,
    }


In [ ]:
# Example: budget the Gemini call from Part 2 (section 1.1.1)
_ = estimate_cost(
    prompt_text=messages[0]["content"] if isinstance(messages, list) else str(messages),
    completion_text=gemini_output,
    model_name="google/gemini-flash-latest",
)


## Part 4: Fine-Tuning (bare minimum)

Two bare-minimum fine-tuning examples: classification fine-tuning with BERT, and generation-model fine-tuning with **LoRA** / **QLoRA**. These are trimmed for demonstration (small samples, few epochs) — not full training runs.

*(Adapted from `Chapter_11_-_Fine-Tuning_BERT.ipynb` and `Chapter_12_-_Fine-tuning_Generation_Models.ipynb`.)*

### 4.1 BERT Fine-Tuning (classification, bare minimum)


In [ ]:
from datasets import load_dataset

# Prepare data and splits
tomatoes = load_dataset("rotten_tomatoes")
train_data, test_data = tomatoes["train"], tomatoes["test"]


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load Model and Tokenizer
model_id = "bert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_id)


In [ ]:
from transformers import DataCollatorWithPadding

# Pad to the longest sequence in the batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def preprocess_function(examples):
   """Tokenize input data"""
   return tokenizer(examples["text"], truncation=True)

# Tokenize train/test data (small subset for a quick demo)
tokenized_train = train_data.select(range(200)).map(preprocess_function, batched=True)
tokenized_test = test_data.select(range(100)).map(preprocess_function, batched=True)


In [ ]:
from transformers import TrainingArguments, Trainer

# Training arguments for a quick demo run
training_args = TrainingArguments(
   "model",
   learning_rate=2e-5,
   per_device_train_batch_size=16,
   per_device_eval_batch_size=16,
   num_train_epochs=1,
   weight_decay=0.01,
   save_strategy="no",
   report_to="none"
)

# Trainer which executes the training process
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   tokenizer=tokenizer,
   data_collator=data_collator,
)

trainer.train()
trainer.evaluate()


### 4.2 Generation Model Fine-Tuning with LoRA / QLoRA (bare minimum)

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

# Load a tokenizer to use its chat template
template_tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

def format_prompt(example):
    """Format the prompt using the <|user|> template TinyLlama uses"""
    chat = example["messages"]
    prompt = template_tokenizer.apply_chat_template(chat, tokenize=False)
    return {"text": prompt}

# Small subset for a quick demo (original tutorial uses 3,000 examples)
dataset = (
    load_dataset("HuggingFaceH4/ultrachat_200k", split="test_sft")
      .shuffle(seed=42)
      .select(range(200))
)
dataset = dataset.map(format_prompt)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# 4-bit quantization configuration - the "Q" in QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,             # Use 4-bit precision model loading
    bnb_4bit_quant_type="nf4",     # Quantization type
    bnb_4bit_compute_dtype="float16",  # Compute dtype
    bnb_4bit_use_double_quant=True,    # Apply nested quantization
)

# Load the model to train on the GPU
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",

    # Leave this out for regular LoRA (no quantization) instead of QLoRA
    quantization_config=bnb_config,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = "<PAD>"
tokenizer.padding_side = "left"


In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

# LoRA configuration
peft_config = LoraConfig(
    lora_alpha=32,      # LoRA scaling
    lora_dropout=0.1,   # Dropout for LoRA layers
    r=64,               # Rank
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=      # Layers to target
     ['k_proj', 'gate_proj', 'v_proj', 'up_proj', 'q_proj', 'o_proj', 'down_proj']
)

# Prepare model for training
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)


In [ ]:
from transformers import TrainingArguments

output_dir = "./results"

# Training arguments for a quick demo run
training_arguments = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    num_train_epochs=1,
    logging_steps=10,
    fp16=True,
    gradient_checkpointing=True
)


In [ ]:
from trl import SFTTrainer

# Supervised fine-tuning with the LoRA/QLoRA config
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=training_arguments,
    max_seq_length=512,

    # Leave this out for regular SFT (no LoRA)
    peft_config=peft_config,
)

# Train model
trainer.train()

# Save QLoRA weights
trainer.model.save_pretrained("TinyLlama-1.1B-qlora")


In [ ]:
from peft import AutoPeftModelForCausalLM

model = AutoPeftModelForCausalLM.from_pretrained(
    "TinyLlama-1.1B-qlora",
    low_cpu_mem_usage=True,
    device_map="auto",
)

# Merge LoRA weights into the base model
merged_model = model.merge_and_unload()


In [ ]:
from transformers import pipeline

# Use our predefined prompt template
prompt = """<|user|>
Tell me something about Large Language Models.</s>
<|assistant|>
"""

# Run our instruction-tuned model
pipe = pipeline(task="text-generation", model=merged_model, tokenizer=tokenizer)
print(pipe(prompt)[0]["generated_text"])
